# Gold Layer

## Objective

The Gold layer contains business-ready datasets created from the repaired Silver layer.

Responsibilities:

- Read repaired Silver datasets
- Generate business KPIs
- Create analytical datasets
- Store Gold tables

# Import Libraries

In [0]:
import os
import pandas as pd

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Load Configuration

In [0]:
from utils.config import *

# Load Latest Silver Batch

In [0]:
silver_batches = sorted(os.listdir(SILVER_PATH))

LATEST_BATCH = silver_batches[-1]

print("Latest Silver Batch:")
print(LATEST_BATCH)

# Build Silver Path

In [0]:
LATEST_SILVER_PATH = os.path.join(
    SILVER_PATH,
    LATEST_BATCH
)

print(LATEST_SILVER_PATH)

#Load Repaired Silver Datasets

In [0]:
bus_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_gps")
)

emergency_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency")
)

print("=" * 60)

print("Bus Silver :", bus_silver_df.count())
print("Emergency Silver :", emergency_silver_df.count())

#Create Gold Output Path

In [0]:
from datetime import datetime

GOLD_BATCH = datetime.now().strftime("%Y%m%d_%H%M%S")

GOLD_BATCH_PATH = os.path.join(
    GOLD_PATH,
    GOLD_BATCH
)

os.makedirs(GOLD_BATCH_PATH, exist_ok=True)

print(GOLD_BATCH_PATH)

#Emergency Incident Summary by Zone

In [0]:
gold_incident_summary_df = (
    emergency_silver_df
    .groupBy("zone")
    .agg(
        count("*").alias("total_incidents"),
        avg("severity").alias("avg_severity"),
        avg("response_time").alias("avg_response_time")
    )
    .orderBy("zone")
)

gold_incident_summary_df.show(truncate=False)

#Save Emergency Incident Summary

In [0]:
gold_incident_summary_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "emergency_incident_summary"
    )
)
print("✅ Emergency Incident Summary saved successfully.")

#Incident Status Summary

In [0]:
gold_status_summary_df = (
    emergency_silver_df
    .groupBy("status")
    .agg(
        count("*").alias("total_incidents"),
        avg("response_time").alias("avg_response_time")
    )
    .orderBy("status")
)

gold_status_summary_df.show(truncate=False)

#Save Incident Status Summary

In [0]:
gold_status_summary_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "incident_status_summary"
    )
)

print("✅ Incident Status Summary saved successfully.")

# Zone Severity Analysis

In [0]:
gold_zone_severity_df = (
    emergency_silver_df
    .groupBy("zone")
    .agg(
        avg("severity").alias("avg_severity"),
        max("severity").alias("max_severity"),
        min("severity").alias("min_severity"),
        count("*").alias("total_incidents")
    )
    .orderBy(desc("avg_severity"))
)

gold_zone_severity_df.show(truncate=False)

# Save Zone Severity Analysis

In [0]:
gold_zone_severity_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "zone_severity_analysis"
    )
)
print("✅ Zone Severity Analysis saved successfully.")

# Response Time Analysis by Incident Type

In [0]:
gold_response_time_df = (
    emergency_silver_df
    .groupBy("incident_type")
    .agg(
        avg("response_time").alias("avg_response_time"),
        max("response_time").alias("max_response_time"),
        min("response_time").alias("min_response_time"),
        count("*").alias("total_incidents")
    )
    .orderBy("avg_response_time")
)
gold_response_time_df.show(truncate=False)

# Save Response Time Analysis

In [0]:
gold_response_time_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "response_time_analysis"
    )
)
print("✅ Response Time Analysis saved successfully.")

# Bus Delay Summary

In [0]:
gold_bus_delay_df = (
    bus_silver_df
    .groupBy("route_id")
    .agg(
        count("*").alias("total_buses"),
        avg("delay_minutes").alias("avg_delay"),
        max("delay_minutes").alias("max_delay"),
        min("delay_minutes").alias("min_delay")
    )
    .orderBy(desc("avg_delay"))
)

gold_bus_delay_df.show(truncate=False)

#Save Bus Delay Summary

In [0]:
gold_bus_delay_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "bus_delay_summary"
    )
)
print("✅ Bus Delay Summary saved successfully.")

#AI Repair Performance Report

In [0]:
gold_ai_summary_df = (
    emergency_silver_df
    .groupBy("repair_status")
    .count()
)

gold_ai_summary_df.show(truncate=False)

#Save AI Repair Report

In [0]:
gold_ai_summary_df.write.mode("overwrite").parquet(
    os.path.join(
        GOLD_BATCH_PATH,
        "ai_repair_summary"
    )
)
print("✅ AI Repair Summary saved successfully.")

#Gold Pipeline Summary

In [0]:
print("=" * 70)
print("SMART CITY GOLD LAYER")
print("=" * 70)

print(f"Emergency Records       : {emergency_silver_df.count()}")
print(f"Bus Records             : {bus_silver_df.count()}")

print(f"Gold Tables Created     : 6")

print("- Emergency Incident Summary")
print("- Incident Status Summary")
print("- Incident Type Summary")
print("- Zone Severity Analysis")
print("- Response Time Analysis")
print("- Bus Delay Summary")

print("=" * 70)
print("GOLD PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

# Verify Gold Folder

In [0]:
print("Gold Batch Path:")
print(GOLD_BATCH_PATH)

print("\nGold Tables:")

for folder in sorted(os.listdir(GOLD_BATCH_PATH)):
    print(folder)